In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

products_df = spark.table(
    f"{catalog}.ecommerce_bronze.products"
)

products_df.printSchema()
products_df.show(5, truncate=False)

In [0]:
#check nulls 

products_df.select([
    F.count(F.when(products_df[c].isNull(), 1)).alias(c)
    for c in products_df.columns
]).show()

In [0]:
silver_products = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn("product_id", F.trim(products_df["product_id"]))
    .withColumn(
        "product_category_name",
        F.trim(products_df["product_category_name"])
    )
    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    )
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )
)

In [0]:
silver_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/silver/products"

silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.products
USING DELTA
LOCATION '{silver_path}'
""")

In [0]:
#verify

spark.table(
    f"{catalog}.ecommerce_silver.products"
).show(5, truncate=False)